# Magnetic Field Evolution in Satellite Galaxies (IllustrisTNG)

In this notebook, we analyze how the magnetic field evolves over cosmic time
for satellite galaxies using IllustrisTNG merger trees.

We will:
- Extract magnetic field data from merger trees
- Convert snapshot number → redshift
- Plot magnetic field vs redshift
- Wrap into reusable functions for multiple galaxies

In [12]:
import os
os.getcwd()

'/workspaces/Numerical_methods_Project/TNGWorkshop'

In [13]:
import sys
sys.executable


'/usr/local/python/3.12.1/bin/python'

In [14]:
!{sys.executable} -m pip install numpy h5py matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Connection to TNG Data

In [15]:
import requests

headers = {"api-key": "f47c8519062d199613c99b12244d62ab"}
r = requests.get('https://www.tng-project.org/api/', headers=headers)
print(r.status_code)
print(r.text)


200
{"simulations":[{"name":"Illustris-1","num_snapshots":134,"url":"http://www.tng-project.org/api/Illustris-1/"},{"name":"Illustris-1-Dark","num_snapshots":136,"url":"http://www.tng-project.org/api/Illustris-1-Dark/"},{"name":"Illustris-2","num_snapshots":136,"url":"http://www.tng-project.org/api/Illustris-2/"},{"name":"Illustris-2-Dark","num_snapshots":136,"url":"http://www.tng-project.org/api/Illustris-2-Dark/"},{"name":"Illustris-3","num_snapshots":136,"url":"http://www.tng-project.org/api/Illustris-3/"},{"name":"Illustris-3-Dark","num_snapshots":136,"url":"http://www.tng-project.org/api/Illustris-3-Dark/"},{"name":"TNG100-1","num_snapshots":100,"url":"http://www.tng-project.org/api/TNG100-1/"},{"name":"TNG100-1-Dark","num_snapshots":100,"url":"http://www.tng-project.org/api/TNG100-1-Dark/"},{"name":"TNG100-2","num_snapshots":100,"url":"http://www.tng-project.org/api/TNG100-2/"},{"name":"TNG100-2-Dark","num_snapshots":100,"url":"http://www.tng-project.org/api/TNG100-2-Dark/"},{"na

## Setup and Imports
This notebook analyzes the evolution of magnetic field strength in a satellite galaxy using TNG simulation data. We will track the magnetic field across time using merger trees and plot it as a function of redshift.

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

from astropy.cosmology import Planck15
import astropy.units as u

import iapi_TNG_MD as iapi  # same as workshop

## Select Satellite Galaxy
We begin by selecting a satellite galaxy and defining the simulation we are working with.

In [17]:
###specify which simulation you want to explore###
sim='TNG100-1'

baseUrl = 'https://www.tng-project.org/api/'
sim = 'TNG100-1'
simUrl = baseUrl + sim

my_gal = 565261  # example galaxy
snapnum = 99     # z = 0

dirc= '/workspaces/Numerical_methods_Project/TNGWorkshop'

## Load Merger Tree

To study evolution over time, we use merger trees. These allow us to trace the main progenitor of a galaxy across snapshots, giving us its history before and after becoming a satellite.

In [18]:
subTreeFile = iapi.gettree(snapnum, subID)
print(subTreeFile)

NameError: name 'subID' is not defined

## Step 3: Extract Magnetic Field and Snapshots

We extract:
- SubhaloBfldDisk → magnetic field in disk
- SubhaloBfldHalo → magnetic field in halo
- SnapNum → used to compute redshift

In [24]:
with h5py.File(subTreeFile, 'r') as f:
    snaps = np.array(f['SnapNum'])
    B_disk = np.array(f['SubhaloBfldDisk'])
    B_halo = np.array(f['SubhaloBfldHalo'])

NameError: name 'subTreeFile' is not defined

## Step 4: Convert Snapshot → Redshift

In [23]:
z = iapi.getredshift(snaps)

NameError: name 'snaps' is not defined

## Step 5: Sort Data Properly

Important:
- Merger trees go from present → past
- We sort by redshift for a clean plot

In [22]:
order = np.argsort(z)

z = z[order]
B_disk = B_disk[order]
B_halo = B_halo[order]

NameError: name 'z' is not defined

## Step 6: Plot Magnetic Field Evolution

In [21]:
plt.figure()

plt.plot(z, B_disk, label='Disk Magnetic Field')
plt.plot(z, B_halo, label='Halo Magnetic Field')

plt.xlabel('Redshift')
plt.ylabel('Magnetic Field Strength')
plt.yscale('log')

plt.gca().invert_xaxis()  # early universe on right

plt.legend()
plt.title('Magnetic Field Evolution')

plt.show()

NameError: name 'z' is not defined

<Figure size 640x480 with 0 Axes>

## Step 7: Wrap into a Function

This allows us to reuse the same analysis for multiple satellite galaxies.

In [20]:
def plot_magnetic_field(subID, sim='TNG100-1', snapnum=99):
    """
    Plots magnetic field evolution for a given subhalo.
    """

    # get merger tree
    subTreeFile = iapi.gettree(snapnum, subID)

    # load data
    with h5py.File(subTreeFile, 'r') as f:
        snaps = np.array(f['SnapNum'])
        B_disk = np.array(f['SubhaloBfldDisk'])
        B_halo = np.array(f['SubhaloBfldHalo'])
    
    B_disk[B_disk <= 0] = np.nan
    B_halo[B_halo <= 0] = np.nan

    # convert to redshift
    z = iapi.getredshift(snaps)

    # sort
    order = np.argsort(z)
    z = z[order]
    B_disk = B_disk[order]
    B_halo = B_halo[order]

    # plot
    plt.figure()
    plt.plot(z, B_disk, label='Disk')
    plt.plot(z, B_halo, label='Halo')

    plt.xlabel('Redshift')
    plt.ylabel('Magnetic Field')
    plt.yscale('log')
    plt.gca().invert_xaxis()

    plt.title(f'Subhalo {subID}')
    plt.legend()
    plt.show()

## Getting Stellar mass and flag

In [ ]:
stellar_mass = iapi.getSubhaloField(
    'SubhaloMassType',
    simulation=sim,
    fileName=dirc+'catalogs/SubhaloMassType',
    rewriteFile=0
)[:,4]   # stars = index 4

flag = iapi.getSubhaloField(
    'SubhaloFlag',
    simulation=sim,
    fileName=dirc+'catalogs/SubhaloFlag',
    rewriteFile=0
)

## Step 8: Select Satellite Galaxies

In [19]:
 # total number of subhalos
subID = np.arange(len(stellar_mass))

# get group number for each subhalo
subhalo_grnr = iapi.getSubhaloField(
    'SubhaloGrNr',
    simulation=sim,
    fileName=dirc+'catalogs/SubhaloGrNr',
    rewriteFile=0
)

# get central galaxy index for each halo
group_firstsub = iapi.getHaloField(
    'GroupFirstSub',
    simulation=sim,
    fileName=dirc+'catalogs/GroupFirstSub',
    rewriteFile=0
)

# map each subhalo → its central galaxy
central_ids = group_firstsub[subhalo_grnr]

# satellite condition
satellite_mask = subID != central_ids

# combine with your existing filters
mask = (flag == 1) & (stellar_mass > 1e8) & satellite_mask

# final satellite list
satellite_IDs = subID[mask]

print("Number of satellites:", len(satellite_IDs))

NameError: name 'stellar_mass' is not defined

## Step 9: Apply to Multiple Galaxies

In [ ]:
# Example: run for multiple satellites
subIDs = [565261, 123456, 789012]  # replace with real IDs

for sid in subIDs:
    plot_magnetic_field(sid)

#better version
for sid in satellite_IDs[:5]:   # start small to test
    plot_magnetic_field(sid)